### Currently (June 2026), I am finding that critic networks in the rl algorithms may be having difficulty generalizing across patient states, and this is making it difficult for the actors to identify actions that lead to higher Q values. 

I suspect this may be due to the high dimensionality of the transformer latents I am using as the input states to the RL training loop - these latents come from both dynamic deephit (dim 512), and the behavioural clone (dim 256).

This notebook uses SVD to determine the number of components to explain most of the variance, and then keeps only the minimum number of essesntial dimensions 

In [96]:
import os
import h5py
import torch

In [97]:
dynamic_deephit_predictions_path = "/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/"
bc_transformer_final_model_dir = "/home/jake/Documents/AI_Clinician/models/bc_transformer/choc/"
cohort = "sepsis"
actions = "intake-vis"
split = 0

## compress dynamic deephit latents

In [98]:
dynamic_deephit_predictions_path = os.path.join(dynamic_deephit_predictions_path, 
    "five_splits", f"split_{0}", f"{cohort}", f"{actions}", "final_model")

# find the folder inside the final_model directory
final_model_dir = os.listdir(dynamic_deephit_predictions_path)[0]
print(final_model_dir)

choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260326_040032_999109_job2024209.pbs-7


In [99]:
dynamic_deephit_train_predictions_path = os.path.join(dynamic_deephit_predictions_path,final_model_dir,
    "train_set_predictions.h5")
print(dynamic_deephit_train_predictions_path)

/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/final_model/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260326_040032_999109_job2024209.pbs-7/train_set_predictions.h5


In [100]:
# load the h5 file
with h5py.File(dynamic_deephit_train_predictions_path, "r") as f:
    dynamic_deephit_latents_train = f["model_outputs"][:]
print(dynamic_deephit_latents_train.shape)

(2488216, 512)


In [101]:

latents_tensor_train = torch.tensor(dynamic_deephit_latents_train, dtype=torch.float32)

# Center
latent_tensor_train_mean = latents_tensor_train.mean(dim=0)

In [102]:
latents_tensor_train -= latent_tensor_train_mean

U, S, Vh = torch.svd_lowrank(latents_tensor_train, q=200)

explained = (S**2).cumsum(0) / (S**2).sum()

for k in [10, 20, 30, 50, 100, 200]:
    print(f"Top {k:3d} components: {explained[k-1]*100:.1f}%")

Top  10 components: 57.5%
Top  20 components: 79.9%
Top  30 components: 91.4%
Top  50 components: 96.1%
Top 100 components: 98.5%
Top 200 components: 100.0%


In [103]:
n_dim = 100
compressed_latents_train = latents_tensor_train @ Vh[:, :n_dim]
compressed_latents_train.shape

torch.Size([2488216, 100])

In [104]:
with h5py.File(dynamic_deephit_train_predictions_path, "a") as f:  # "a" = append mode
    if "compressed_latents" in f:
        del f["compressed_latents"]  # delete existing dataset if it exists
    f.create_dataset("compressed_latents", data=compressed_latents_train.cpu().numpy())

In [105]:
# load the compressed latents to verify
with h5py.File(dynamic_deephit_train_predictions_path, "r") as f:
    loaded_compressed_latents_train = f["compressed_latents"][:]
print(loaded_compressed_latents_train.shape)

(2488216, 100)


In [106]:
# load the original latents to verify
with h5py.File(dynamic_deephit_train_predictions_path, "r") as f:
    loaded_latents_train = f["model_outputs"][:]
print(loaded_latents_train.shape)

(2488216, 512)


In [107]:
# compress val and test
dynamic_deephit_val_predictions_path = os.path.join(dynamic_deephit_predictions_path,final_model_dir,
    "val_set_predictions.h5")
dynamic_deephit_test_predictions_path = os.path.join(dynamic_deephit_predictions_path, final_model_dir,
    "test_set_predictions.h5")

for path in [dynamic_deephit_val_predictions_path, dynamic_deephit_test_predictions_path]:
    with h5py.File(path, "r") as f:
        latents = f["model_outputs"][:]
    latents_tensor = torch.tensor(latents, dtype=torch.float32)
    latents_tensor -= latent_tensor_train_mean
    compressed_latents = latents_tensor @ Vh[:, :n_dim]
    with h5py.File(path, "a") as f:
        if "compressed_latents" in f:
            del f["compressed_latents"]  # delete existing dataset if it exists
        f.create_dataset("compressed_latents", data=compressed_latents.cpu().numpy())                                                    

## compress behavioural clone latents

In [88]:
bc_clone_predictions_path = os.path.join(bc_transformer_final_model_dir, 
    "five_splits", f"split_{0}", f"{cohort}", f"{actions}", "final_model")

# find the folder inside the final_model directory
final_model_dir = os.listdir(bc_clone_predictions_path)[0]
print(final_model_dir)

unknown_cx3-20-3_20260531-160828_79f0d952


In [89]:
bc_clone_train_predictions_path = os.path.join(bc_clone_predictions_path,final_model_dir,
    "train_set_predictions.h5")
print(bc_clone_train_predictions_path)

/home/jake/Documents/AI_Clinician/models/bc_transformer/choc/five_splits/split_0/sepsis/intake-vis/final_model/unknown_cx3-20-3_20260531-160828_79f0d952/train_set_predictions.h5


In [90]:
# load the h5 file
with h5py.File(bc_clone_train_predictions_path, "r") as f:
    bc_clone_latents_train = f["model_outputs"][:]
print(bc_clone_latents_train.shape)

(715424, 256)


In [91]:
latents_tensor_train = torch.tensor(bc_clone_latents_train, dtype=torch.float32)

# Center
latent_tensor_train_mean = latents_tensor_train.mean(dim=0)
latents_tensor_train -= latent_tensor_train_mean

In [92]:
U, S, Vh = torch.svd_lowrank(latents_tensor_train, q=256)

explained = (S**2).cumsum(0) / (S**2).sum()

for k in [8, 16, 32, 64, 128, 256]:
    print(f"Top {k:3d} components: {explained[k-1]*100:.1f}%")

Top   8 components: 97.3%
Top  16 components: 98.7%
Top  32 components: 99.3%
Top  64 components: 99.7%
Top 128 components: 99.9%
Top 256 components: 100.0%


In [93]:
n_dim = 10
compressed_latents_train = latents_tensor_train @ Vh[:, :n_dim]
compressed_latents_train.shape

torch.Size([715424, 10])

In [94]:
with h5py.File(bc_clone_train_predictions_path, "a") as f:  # "a" = append mode
    if "compressed_latents" in f:
        del f["compressed_latents"]  # delete existing dataset if it exists
    f.create_dataset("compressed_latents", data=compressed_latents_train.cpu().numpy())

In [95]:
# compress val and test
bc_val_predictions_path = os.path.join(bc_clone_predictions_path,final_model_dir,
    "val_set_predictions.h5")
bc_test_predictions_path = os.path.join(bc_clone_predictions_path, final_model_dir,
    "test_set_predictions.h5")

for path in [bc_val_predictions_path, bc_test_predictions_path]:
    with h5py.File(path, "r") as f:
        latents = f["model_outputs"][:]
    latents_tensor = torch.tensor(latents, dtype=torch.float32)
    latents_tensor -= latent_tensor_train_mean
    compressed_latents = latents_tensor @ Vh[:, :n_dim]
    with h5py.File(path, "a") as f:
        if "compressed_latents" in f:
            del f["compressed_latents"]  # delete existing dataset if it exists
        f.create_dataset("compressed_latents", data=compressed_latents.cpu().numpy())     